## Weekend Model for Quant Position Interview
 This code was written in response to an problem posed for the purpose of an interview for a quant position at a bank. I wrote the code and created the corresponding presentation in a weekend. I received an offer following the presentation of this code and the outputs generated.

 The dataset is short and lacking details, and one of the first tests was whether or not I would overcomplicate things by created a ML model or more complex model, which I did not.

In [ ]:
import warnings
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
from statsmodels.tools.eval_measures import rmse
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
OUTDIR = '/content/drive/MyDrive/Job/weekend_model/'
INFILE = '/content/drive/MyDrive/Job/weekend_model/Auto Loan Time Series Variables.xlsx'

SIGNS = {
    'RealGDP': -1,
    'CPI': 1,
    'HPI': -1,
    'Unemployment': 1,
    'RealDisposablePersonalIncome': -1,
    'LightWeightVehicleSales': 0,
    'ManheimUsedVehicleValueIndex': 1
}

# ---------------------------------------------------------
# CORE FUNCTIONS
# ---------------------------------------------------------

def transform_feature(df, col, run_params):
    """Applies rolling means and transformations to a single column."""
    ms = run_params.get('ms', 3)
    lags = run_params.get('lags', 5)
    maxlag = run_params.get('maxlag', 8)

    transforms = ['qoq', 'dq', 'yoy', 'ldq', 'lvl']
    tlag = {'qoq': 1, 'dq': 1, 'yoy': 4, 'ldq': 1, 'lvl': 0}
    results = {}

    for lag in range(lags):
        for m in range(1, ms + 1):
            for trans in transforms:
                if (m - 1 + lag + tlag[trans]) <= maxlag:
                    now = df[col].rolling(int(m)).mean()
                    qm1 = now.shift(1)
                    ym1 = now.shift(4)

                    funcs = {
                        'qoq': (now - qm1) / qm1,
                        'dq':  (now - qm1),
                        'yoy': (now - ym1) / ym1,
                        'ldq': np.log(now) - np.log(qm1),
                        'lvl': now
                    }

                    newcol = f'{col}_{trans}_l{lag}_m{m}'
                    results[newcol] = funcs[trans].shift(int(lag))

    return pd.DataFrame(results)

def standardize_df(df):
    """Standardizes all columns in a DataFrame."""
    scaler = StandardScaler()
    return pd.DataFrame(scaler.fit_transform(df), index=df.index, columns=df.columns)

def get_stationary_features(df, p_val_threshold=0.05):
    """Runs ADF tests and returns only features that pass stationarity."""
    p_values = {col: adfuller(df[col].dropna())[1] for col in df.columns}
    res_df = pd.Series(p_values, name='p_adf').to_frame()
    passing = res_df[res_df['p_adf'] < p_val_threshold]

    print(f"{len(res_df)} variables tested | {len(passing)} stationary variables")
    return passing.index.tolist()

def check_signs(model, expected_signs):
    """Checks if model parameters match the intuitively expected signs."""
    for var, coef in model.params.items():
        if var == 'const':
            continue
        root = var.split('_')[0]
        expected = expected_signs.get(root, 0)

        # If an expected sign is defined (non-zero) and doesn't match the coefficient's sign
        if expected != 0 and np.sign(coef) != expected:
            return False
    return True

def univariate_selection(df, features, target, expected_signs):
    """Runs univariate OLS and returns features that are significant and intuitive."""
    results = []
    for factor in features:
        X = sm.add_constant(df[factor])
        model = sm.OLS(df[target], X).fit()

        if check_signs(model, expected_signs):
            results.append({
                'factor': factor,
                'root_var': factor.split('_')[0],
                'p_value': model.pvalues[factor],
                'adj_r2': model.rsquared_adj
            })

    return pd.DataFrame(results).set_index('factor')

def get_vif_df(X):
    """Calculates Variance Inflation Factor for a feature matrix."""
    vifs = {X.columns[i]: vif(X.values, i) for i in range(X.shape[1]) if X.columns[i] != 'const'}
    return pd.Series(vifs, name='VIF')

def format_model_results(model, exog_df):
    """Replaces HTML parsing with direct extraction of statsmodels parameters."""
    res_df = pd.DataFrame({
        'Coef': model.params,
        'P-value': model.pvalues
    })

    # Merge VIF
    vif_series = get_vif_df(exog_df)
    res_df = res_df.join(vif_series).drop(index='const', errors='ignore')

    # Parse feature components for final output table
    res_df['RootVariable'] = [idx.split('_')[0] for idx in res_df.index]
    res_df['Transform'] = [idx.split('_')[1] for idx in res_df.index]
    res_df['Lag'] = [idx.split('_')[2][1] for idx in res_df.index]
    res_df['MovingAvg'] = [idx.split('_')[3][1] for idx in res_df.index]

    return res_df[['RootVariable', 'Transform', 'Lag', 'MovingAvg', 'Coef', 'P-value', 'VIF']].set_index('RootVariable')

def run_variable_selection(df, target, feasible_vars, run_params):
    """Optimized feature combination search using itertools product and combinations."""
    r2min = run_params['r2min']
    test_size = run_params['test']
    # Train/Test Split
    X_train, X_test, Y_train, Y_test = train_test_split(df.drop(columns=[target]), df[target], test_size=test_size, random_state=1)
    X_train = sm.add_constant(X_train)
    X_test = sm.add_constant(X_test)

    # Group feasible variables by their root variable to ensure uniqueness in combinations
    root_groups = {}
    for factor, row in feasible_vars.iterrows():
        root_groups.setdefault(row['root_var'], []).append(factor)

    feasible_roots = list(root_groups.keys())
    # Pick N distinct roots, then create Cartesian products of their variations

    root_combinations = (
        list(itertools.combinations(feasible_roots, 2)) +
        list(itertools.combinations(feasible_roots, 3))
    )

    passed_models = []
    checked = 0

    for combo in root_combinations:
        feature_lists = [root_groups[root] for root in combo]

        # Test every variation of the chosen root combination
        for feature_set in itertools.product(*feature_lists):
            checked += 1
            features_with_const = list(feature_set) + ['const']

            model = sm.OLS(Y_train, X_train[features_with_const]).fit()
            max_p = model.pvalues.drop('const').max()

            # Check p-value and R-squared constraints
            if max_p < 0.05 and model.rsquared_adj > r2min:
                if check_signs(model, SIGNS):
                    # Store metrics
                    passed_models.append({
                        'features': feature_set,
                        'R2_adj_train': model.rsquared_adj,
                        'model_obj': model
                    })

    print(f"Tested {checked} combinations. {len(passed_models)} passed constraints.")

    if not passed_models:
        return pd.DataFrame()

    # Sort models by Train Adj R2 descending
    pm_df = pd.DataFrame(passed_models).sort_values('R2_adj_train', ascending=False)
    return pm_df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---------------------------------------------------------
# 1. READ AND CLEAN DATA
# ---------------------------------------------------------
def load_and_clean_data(filepath):
    """Loads dataset, cleans column names, and sets appropriate index."""
    raw_df = pd.read_excel(filepath, index_col=0)

    # Clean column names (remove spaces, hyphens, colons)
    clean_cols = [str(col).replace(' ', '').replace('-', '').replace(':', '') for col in raw_df.columns]

    # Skip metadata rows (first two rows) and assign cleaned names
    df = raw_df.iloc[2:].copy()
    df.columns = clean_cols
    df.index.name = 'Period'

    # Convert all values to numeric
    df = df.apply(pd.to_numeric, errors='coerce')

    target_col = clean_cols[-1]
    feature_cols = clean_cols[:-1]

    return df, feature_cols, target_col

# ---------------------------------------------------------
# 2. PROFILE VARIABLES (PLOTTING)
# ---------------------------------------------------------
def plot_raw_profiles(df, features, target, outdir):
    """Plots all features against the target variable in a dynamic grid."""
    cols_count = 3
    rows_count = int(np.ceil(len(features) / cols_count))

    fig, axes = plt.subplots(rows_count, cols_count, figsize=(18, rows_count * 4))
    axes = axes.flatten() # Flatten 2D axes array for easy 1D iteration

    for idx, col in enumerate(features):
        ax1 = axes[idx]
        ax2 = ax1.twinx()

        p1 = ax1.plot(df.index, df[target], color='blue', label='COR (Target)')
        p2 = ax2.plot(df.index, df[col], color='darkorange', label=col)

        ax1.set_ylabel('Charge Off Rate')
        ax2.set_ylabel(col)
        ax1.set_title(col)
        ax1.legend(p1 + p2, ['COR', 'Factor'], loc='upper left')

    # Hide any unused subplots in the grid
    for ax in axes[len(features):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.savefig(f'{outdir}raw_plots.png')
    plt.close()

# ---------------------------------------------------------
# 3. MAIN EXECUTION PIPELINE
# ---------------------------------------------------------
if __name__ == "__main__":

    # Run configuration
    run_params = {
        'outfile': 'example',
        'r2min': 0.2,
        'maxlag': 5,
        'dropcols': ['Unemployment', 'CPI'],
        'test': 0.2
    }

    print("1. Loading and cleaning data...")
    al_df, vcols, target = load_and_clean_data(INFILE)

    print("2. Profiling raw variables...")
    plot_raw_profiles(al_df, vcols, target, OUTDIR)

    print("3. Transforming and standardizing features...")
    transformed_cols = [transform_feature(al_df, col, run_params) for col in vcols]
    transformed_df = pd.concat(transformed_cols, axis=1)
    standardized_features = standardize_df(transformed_df)

    # Combine with target and drop NAs created by lags/rolling means
    mod_data = standardized_features.join(al_df[target]).dropna()

    # Drop explicitly excluded feature roots based on configuration
    cols_to_drop = [col for col in mod_data.columns if any(drop_var in col for drop_var in run_params['dropcols'])]
    mod_data = mod_data.drop(columns=cols_to_drop)

    print(f"   -> Dataset ready with {mod_data.shape[1]-1} candidate feature transformations.")

    print("\n4. Running Stationarity Tests...")
    stationary_features = get_stationary_features(mod_data.drop(columns=[target]), p_val_threshold=0.05)

    print("\n5. Running Univariate Selection...")
    feasible_vars = univariate_selection(mod_data, stationary_features, target, SIGNS)
    print(f"   -> {len(feasible_vars)} variables exhibit intuitive signs and significance.")

    print("\n6. Running Combinatorial Model Search...")
    passing_models_df = run_variable_selection(mod_data, target, feasible_vars, run_params)

    # Output Results
    if not passing_models_df.empty:
        print(f"\nSuccess! Found {len(passing_models_df)} passing models.")

        # ---------------------------------------------------------
        # A. Split details into expanded CSV
        # ---------------------------------------------------------
        expanded_rows = []
        for rank, (_, row) in enumerate(passing_models_df.iterrows(), 1):
            model = row['model_obj']
            features = row['features']

            # Base dictionary with metrics and intercept
            row_dict = {
                'Rank': rank,
                'Adj_R2': round(row['R2_adj_train'], 4),
                'Intercept': model.params['const']
            }

            # Dynamically map up to 3 factors and their coefficients
            for i in range(3):
                factor_col = f'Factor_{i+1}'
                coef_col = f'Coef_{i+1}'

                if i < len(features):
                    feat = features[i]
                    row_dict[factor_col] = feat
                    row_dict[coef_col] = model.params[feat]
                else:
                    row_dict[factor_col] = None
                    row_dict[coef_col] = None

            expanded_rows.append(row_dict)

        detailed_df = pd.DataFrame(expanded_rows)
        out_csv = f"{OUTDIR}{run_params['outfile']}_detailed_models.csv"
        detailed_df.to_csv(out_csv, index=False)
        print(f"   -> Exported expanded model metrics to {out_csv}")

        # ---------------------------------------------------------
        # B. Export Top 3 Specs to Excel & Generate Forecast Charts
        # ---------------------------------------------------------
        top_3_df = passing_models_df.head(3)
        excel_out = f"{OUTDIR}{run_params['outfile']}_Top3_Specs.xlsx"
        print(f"   -> Exporting top 3 model specifications to {excel_out} and generating charts...")

        with pd.ExcelWriter(excel_out) as writer:
            for i, (_, row) in enumerate(top_3_df.iterrows(), 1):
                model = row['model_obj']

                # Extract the subset of data used by this specific model
                exog_features = list(row['features']) + ['const']
                mod_data['const'] = 1
                exog_df = mod_data[exog_features]

                # 1. Write detailed specs to a dedicated Excel sheet
                spec_df = format_model_results(model, exog_df)
                spec_df.to_excel(writer, sheet_name=f'Model_{i}_Ranked')

                # 2. Generate and save Forecast vs Actual Chart
                fig, ax = plt.subplots(figsize=(10, 6))

                actual = mod_data[target]
                forecast = model.predict(exog_df)

                ax.plot(actual.index, actual, color='blue', label='Actual COR', linewidth=2)
                ax.plot(forecast.index, forecast, color='red', label='Forecast COR', linestyle='--', linewidth=2)

                ax.set_title(f"Rank {i} Model: Forecast vs Actual\n(Adj R2: {row['R2_adj_train']:.4f})")
                ax.set_ylabel("Net Charge Off Rate")
                ax.legend(loc='best')
                ax.grid(True, linestyle=':', alpha=0.6)

                plt.tight_layout()
                chart_path = f"{OUTDIR}{run_params['outfile']}_Model_{i}_Forecast.png"
                fig.savefig(chart_path)
                plt.close(fig)

    else:
        print("\nNo models passed the specified constraints. Consider lowering 'r2min' or adjusting features.")

1. Loading and cleaning data...
2. Profiling raw variables...
3. Transforming and standardizing features...
   -> Dataset ready with 265 candidate feature transformations.

4. Running Stationarity Tests...
265 variables tested | 83 stationary variables

5. Running Univariate Selection...
   -> 43 variables exhibit intuitive signs and significance.

6. Running Combinatorial Model Search...
RUN PARAMS: {'outfile': 'example', 'r2min': 0.2, 'maxlag': 5, 'dropcols': ['Unemployment', 'CPI'], 'test': 0.2}
Tested 4246 combinations. 172 passed constraints.

Success! Found 172 passing models.
   -> Exported expanded model metrics to /content/drive/MyDrive/Job/weekend_model/example_detailed_models.csv
   -> Exporting top 3 model specifications to /content/drive/MyDrive/Job/weekend_model/example_Top3_Specs.xlsx and generating charts...
